# Chapter 5 — LiDAR: Point Clouds and Vegetation Structure

Chapters 1–4 looked at the canopy from above with the multispectral camera. This chapter uses the **LiDAR** flown over the same plot on the same day (DJI Zenmuse L2) to look *into* the canopy: how tall it is, how the vegetation is layered, and where the gaps are.

| Section | Topic | Time |
|---|---|---|
| 1–5 | Locate and open the LiDAR: LAS vs COPC, CRS, coverage, attributes | 4:25–4:35 |
| 6 | Subset and filter: levels of detail, classes, height, returns | 4:35–4:50 |
| 7–9 | Visualise structure: 3D view, cross-section, canopy height profile | 4:50–4:55 |
| 10 | Close and resources | 4:55–5:00 |

The point cloud has **339 million points** (a 3.9 GB file). It is stored as a **COPC** (Cloud-Optimised Point Cloud), which lets us read only the points inside the same study area as Chapters 1–4, straight from the workshop STAC catalogue.

**Before starting:** keep this notebook beside `workshop_utils.py`, start with a fresh kernel and run the cells from top to bottom. This chapter needs up to 2 GB of memory, so shut down the kernels of Chapters 1–4 first (**Kernel → Shut Down All Kernel**).

## 1. Load libraries and shared workshop functions

[`laspy`](https://laspy.readthedocs.io) reads LAS, LAZ and COPC files (with `lazrs` for the LAZ compression). The rest are the libraries from earlier chapters.

In [ ]:
import io
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import laspy
from laspy import CopcReader, Bounds
import rasterio
from rasterio.windows import from_bounds
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output

from workshop_utils import (
    ITEM_ID, AOI_BOUNDS, STAC_COLLECTION_URL, get_asset_href,
)

pio.renderers.default = "plotly_mimetype+notebook_connected"
plt.style.use("seaborn-v0_8-whitegrid")
OUTPUT_DIR = Path("outputs")

# The same study area as Chapters 1-4 (left, bottom, right, top) in GDA2020 / MGA 53
LEFT, BOTTOM, RIGHT, TOP = AOI_BOUNDS
print(f"Survey {ITEM_ID} | study area {RIGHT - LEFT:.1f} m x {TOP - BOTTOM:.1f} m")


def style_map(ax, title):
    ax.set_title(title, fontsize=13, weight="bold")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain", useOffset=False)

## 2. Locate the LiDAR in the catalogue

Each survey in the STAC catalogue has several LiDAR assets. **Level 2** (`lvl2`) holds point clouds; **level 3a** (`lvl3a`) holds rasters derived from them: a canopy height model (CHM), digital surface and terrain models (DSM, DTM), hillshades and a canopy-gap mask, plus a processing report.

In [ ]:
import pystac

item = pystac.Collection.from_file(STAC_COLLECTION_URL).get_item(ITEM_ID)
rows = []
for key, asset in item.assets.items():
    if "/lidar/" in asset.href:
        rows.append({
            "asset": key,
            "file": Path(asset.href).name,
            "media type": asset.media_type,
            "size (GB)": round(asset.extra_fields.get("file:size", 0) / 1e9, 3),
        })
pd.DataFrame(rows)

## 3. LAS, LAZ or COPC?

All three use the same **LAS** point format defined by the ASPRS:

- **LAS** (`.las`): uncompressed points usually in scan order.
- **LAZ** (`.laz`): same as .las but with compression.
- **COPC** (`.copc.laz`): a LAZ 1.4 file with points are arranged in an **octree**, a spatial index of nested cubes. The 3D equivalent of a COG's tiles. Like a COG, a COPC is just `.laz` file meeting certain requirements, including a header that provides details on how the file is arranged. There isn't really a `.copc` filetype, `.copc.laz` is just a naming convention. 

Below we read only the **headers** of two catalogue files: the COPC and the decimated LAS. For the LAS we fetch just its first 64 KB with an HTTP range request.

In [ ]:
COPC_URL = get_asset_href(asset_key="pointcloud")
LAS_URL = get_asset_href(asset_key="pointcloud_decimated")

with CopcReader.open(COPC_URL) as reader:       # reads the header and octree index only
    copc_header = reader.header
    copc_info = reader.copc_info

first_bytes = requests.get(LAS_URL, headers={"Range": "bytes=0-65535"}, timeout=60).content
las_header = laspy.LasReader(io.BytesIO(first_bytes)).header


def describe(name, header, size_bytes):
    vlr_ids = [vlr.user_id for vlr in header.vlrs]
    return {
        "file": name,
        "LAS version": str(header.version),
        "point format": header.point_format.id,
        "points": f"{header.point_count:,}",
        "size (GB)": round(size_bytes / 1e9, 2),
        "compressed": header.are_points_compressed if hasattr(header, "are_points_compressed") else "?",
        "VLRs": ", ".join(vlr_ids),
        "spatial index (copc VLR)": "copc" in vlr_ids,
    }


sizes = {k: item.assets[k].extra_fields["file:size"] for k in ("pointcloud", "pointcloud_decimated")}
pd.DataFrame([
    describe(Path(COPC_URL).name, copc_header, sizes["pointcloud"]),
    describe(Path(LAS_URL).name, las_header, sizes["pointcloud_decimated"]),
]).set_index("file").T

The COPC is LAS **1.4**, compressed, and has a `copc` record: we can ask it for any area. The decimated LAS is LAS **1.2**, uncompressed and has no index. To find the points in our study area we would have to download all 1.7 GB of it and throw most away.

The octree is described by a centre, a half-size (the root cube is twice this wide) and a point **spacing** at the root level. Each deeper level halves the spacing, so level 0 is a sparse overview of the whole survey and the deepest level holds the full-density detail.

In [ ]:
print(f"Octree centre (E, N, Z): {np.round(copc_info.center, 1)}")
print(f"Root cube width:         {2 * copc_info.halfsize:.1f} m")
print(f"Point spacing at level 0: {copc_info.spacing:.2f} m (halves at each level)")

## 4. Area of Interest
Every LAS header stores the coordinate reference system (CRS), the scale and offset used to store coordinates as integers, and the bounding box of all points. The CRS matches the multispectral orthomosaic (GDA2020 / MGA zone 53), so the two line up without reprojecting.

The map shows the CHM hillshade for the whole survey (read at low resolution from a COG overview) with our AOI in red.

In [ ]:
crs = copc_header.parse_crs()
mins, maxs = copc_header.mins, copc_header.maxs
survey_area = (maxs[0] - mins[0]) * (maxs[1] - mins[1])

print(f"CRS:     {crs.name} (EPSG:{crs.to_2d().to_epsg()})")
print(f"Scale:   {copc_header.scales} m  (coordinates are stored as integers x scale + offset)")
print(f"Offset:  {copc_header.offsets}")
print(f"Extent:  {maxs[0] - mins[0]:.0f} m x {maxs[1] - mins[1]:.0f} m, Z {mins[2]:.1f} to {maxs[2]:.1f} m")
print(f"Nominal density: {copc_header.point_count / survey_area:,.0f} points/m² "
      "(bounding box, so a slight underestimate)")

with rasterio.open(get_asset_href(asset_key="chm_hillshade")) as src:
    overview = src.read(1, out_shape=(src.height // 16, src.width // 16))
    survey_extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

fig, ax = plt.subplots(figsize=(7, 8), constrained_layout=True)
ax.imshow(overview, cmap="gray", extent=survey_extent)
ax.add_patch(Rectangle((LEFT, BOTTOM), RIGHT - LEFT, TOP - BOTTOM, fill=False, ec="red", lw=2,
                       label="Study area (Chapters 1-5)"))
ax.set_title("Survey coverage (CHM hillshade)", fontsize=14, weight="bold")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.ticklabel_format(style="plain", useOffset=False)
ax.legend(loc="lower right")
plt.show()

## 5. Filtering to the AOI, one octree level at a time

`reader.query(bounds=..., level=n)` returns the points of octree level *n* that fall inside the study area. We read each level separately and keep a `level` column, so later we can switch between a coarse and a full-density cloud without downloading again. All levels together are the complete, full-density cloud for the study area (about 7 million points, 10–20 seconds).

We keep the attributes we need as a `pandas` table, one row per point. Colours are stored as 16-bit values; we scale them to 0–255.

In [ ]:
aoi = Bounds(mins=np.array([LEFT, BOTTOM]), maxs=np.array([RIGHT, TOP]))
start = time.time()
parts = []
with CopcReader.open(COPC_URL) as reader:
    for level in range(20):
        pts = reader.query(bounds=aoi, level=level)
        if len(pts) == 0:
            break
        parts.append(pd.DataFrame({
            "x": np.asarray(pts.x), "y": np.asarray(pts.y), "z": np.asarray(pts.z),
            "hag": np.asarray(pts["HeightAboveGround"], dtype="float32"),
            "classification": np.asarray(pts.classification, dtype="uint8"),
            "return_number": np.asarray(pts.return_number, dtype="uint8"),
            "number_of_returns": np.asarray(pts.number_of_returns, dtype="uint8"),
            "intensity": np.asarray(pts.intensity),
            "red": (np.asarray(pts.red) >> 8).astype("uint8"),
            "green": (np.asarray(pts.green) >> 8).astype("uint8"),
            "blue": (np.asarray(pts.blue) >> 8).astype("uint8"),
            "level": np.uint8(level),
        }))
pc = pd.concat(parts, ignore_index=True)
del parts

aoi_area = (RIGHT - LEFT) * (TOP - BOTTOM)
print(f"Read {len(pc):,} points in {time.time() - start:.1f} s "
      f"({len(pc) / aoi_area:,.0f} points/m², {pc.memory_usage().sum() / 1e6:.0f} MB in memory)")
levels = pc.groupby("level").size().rename("points").to_frame()
levels["cumulative"] = levels["points"].cumsum()
levels["points/m² up to this level"] = (levels["cumulative"] / aoi_area).round(1)
levels

### What attributes does each point have?

Below shows the main **dimensions** (attributes) each point stores, with their type and range in our study area. The file holds a few more, such as scan angle and flight-line flags; `list(copc_header.point_format.dimension_names)` lists them all. Point format 7 is a standard LAS 1.4 format with colour; `HeightAboveGround` is an **extra dimension** added during processing.

In [ ]:
DESCRIPTIONS = {
    "X": "Easting (stored as integer, x scale + offset)",
    "Y": "Northing (stored as integer, x scale + offset)",
    "Z": "Ellipsoidal height (stored as integer, x scale + offset)",
    "intensity": "Strength of the returned pulse; depends on target, range and angle",
    "return_number": "Which echo of the pulse this is (1 = first)",
    "number_of_returns": "How many echoes the pulse produced",
    "classification": "ASPRS class code (2 = ground, 1 = unclassified, ...)",
    "gps_time": "GPS time of the pulse",
    "red": "Colour from the L2's RGB camera",
    "green": "Colour from the L2's RGB camera",
    "blue": "Colour from the L2's RGB camera",
    "HeightAboveGround": "Extra dimension: height above the classified ground (m)",
}

with CopcReader.open(COPC_URL) as reader:
    sample = reader.query(bounds=aoi, level=range(0, 4))   # a small sample, just for ranges
schema = []
for name, description in DESCRIPTIONS.items():   # only the attributes described above
    values = np.asarray(sample[name])
    schema.append({"dimension": name, "type": str(values.dtype),
                   "min": values.min(), "max": values.max(), "description": description})
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(schema).set_index("dimension")

## 6. Subset and filter

A study area of 7 million points is already a "manageable subset" of the 339 million, but you rarely need all of them at once. The controls below filter the cloud we read in Section 5:

- **Max octree level**: lower levels are a thinned, evenly spaced version of the same cloud. Watch the density drop and the map stay recognisable.
- **Classes**: ground (2) and unclassified (1, here mostly vegetation).
- **Height above ground**: e.g. keep only points above 2 m to see the tree crowns.
- **Returns**: first returns hit the top of whatever the pulse met first; last returns get closest to the ground.

The map is a top-down view of the selected points coloured by height above ground (the highest point in each 0.25 m pixel).

In [ ]:
CLASS_NAMES = {1: "Unclassified", 2: "Ground"}
max_level = int(pc["level"].max())


def select_points(max_lvl, classes, hag_range, returns):
    keep = (pc["level"].values <= max_lvl) & np.isin(pc["classification"].values, classes)
    keep &= (pc["hag"].values >= hag_range[0]) & (pc["hag"].values <= hag_range[1])
    if returns == "First":
        keep &= pc["return_number"].values == 1
    elif returns == "Last":
        keep &= pc["return_number"].values == pc["number_of_returns"].values
    return keep


def top_down(keep, res=0.25):
    # Highest HAG per pixel: sort ascending, then later (higher) points overwrite earlier ones.
    ncol, nrow = int((RIGHT - LEFT) / res), int((TOP - BOTTOM) / res)
    col = np.clip(((pc["x"].values[keep] - LEFT) / res).astype(int), 0, ncol - 1)
    row = np.clip(((TOP - pc["y"].values[keep]) / res).astype(int), 0, nrow - 1)
    hag = pc["hag"].values[keep]
    order = np.argsort(hag)
    image = np.full((nrow, ncol), np.nan, dtype="float32")
    image[row[order], col[order]] = hag[order]
    return image


level_w = widgets.IntSlider(value=max_level, min=0, max=max_level, description="Max level")
class_w = widgets.SelectMultiple(options=[(f"{k} {v}", k) for k, v in CLASS_NAMES.items()],
                                 value=[1, 2], description="Classes", rows=2)
hag_w = widgets.FloatRangeSlider(value=[0, float(pc["hag"].max())], min=0,
                                 max=float(np.ceil(pc["hag"].max())), step=0.25,
                                 description="HAG (m)", continuous_update=False)
ret_w = widgets.ToggleButtons(options=["All", "First", "Last"], description="Returns")


def show_subset(max_lvl, classes, hag_range, returns):
    keep = select_points(max_lvl, list(classes), hag_range, returns)
    n = int(keep.sum())
    fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
    ax.set_facecolor("#eef1f4")
    if n:
        im = ax.imshow(top_down(keep), cmap="viridis", extent=[LEFT, RIGHT, BOTTOM, TOP],
                       vmin=0, vmax=float(pc["hag"].max()))
        fig.colorbar(im, ax=ax, label="Height above ground (m)", shrink=0.85)
    style_map(ax, f"{n:,} points ({n / aoi_area:,.0f} per m²)")
    plt.show()


out = widgets.interactive_output(show_subset, {"max_lvl": level_w, "classes": class_w,
                                               "hag_range": hag_w, "returns": ret_w})
display(widgets.VBox([widgets.HBox([level_w, class_w]), widgets.HBox([hag_w, ret_w]), out]))

## 7. Visualise in 3D

A random sample of points (the browser struggles above a few hundred thousand). Drag to rotate, scroll to zoom. Try each colouring:

- **Elevation (Z)**: terrain and canopy together.
- **Height above ground**: the **normalised** cloud. Terrain is removed, so equal colours mean equal vegetation height anywhere in the plot. This plot is nearly flat, so the difference is small here, but on a slope it is large.
- **Class**: ground (2) vs unclassified (1).
- **RGB**: the colour the L2's camera saw.

Coordinates are shown in metres from the south-west corner of the study area. Heights are exaggerated by the factor `VERTICAL_EXAGGERATION`.

In [ ]:
N_3D = 120_000
CLASS_COLOURS = {1: "#31a354", 2: "#a6761d"}
VERTICAL_EXAGGERATION = 2.0

rng = np.random.default_rng(42)
s3d = pc.iloc[rng.choice(len(pc), size=min(N_3D, len(pc)), replace=False)]


def plot_3d(colour_by, z_mode):
    z = s3d["hag"] if z_mode == "Height above ground" else s3d["z"] - pc["z"].min()
    marker = dict(size=1.5)
    if colour_by == "Class":
        marker["color"] = s3d["classification"].map(CLASS_COLOURS)
        hover = s3d["classification"].map(CLASS_NAMES)
    elif colour_by == "RGB":
        marker["color"] = [f"rgb({r},{g},{b})" for r, g, b in s3d[["red", "green", "blue"]].values]
        hover = None
    else:
        values = s3d["z"] if colour_by == "Elevation (Z)" else s3d["hag"]
        marker.update(color=values, colorscale="Viridis", colorbar=dict(title=colour_by, len=0.6))
        hover = values.round(2).astype(str)
    fig = go.Figure(go.Scatter3d(x=s3d["x"] - LEFT, y=s3d["y"] - BOTTOM, z=z, mode="markers",
                                 marker=marker, text=hover, hoverinfo="text" if hover is not None else "skip"))
    span = max(RIGHT - LEFT, TOP - BOTTOM)
    fig.update_layout(
        height=650, margin=dict(l=0, r=0, t=40, b=0),
        title=f"{len(s3d):,} sampled points, coloured by {colour_by.lower()}",
        scene=dict(xaxis_title="East (m)", yaxis_title="North (m)", zaxis_title=z_mode + " (m)",
                   aspectmode="manual",
                   aspectratio=dict(x=(RIGHT - LEFT) / span, y=(TOP - BOTTOM) / span,
                                    z=VERTICAL_EXAGGERATION * float(z.max()) / span)),
    )
    fig.show()


widgets.interact(
    plot_3d,
    colour_by=widgets.Dropdown(options=["Height above ground", "Elevation (Z)", "Class", "RGB"], description="Colour by"),
    z_mode=widgets.ToggleButtons(options=["Height above ground", "Elevation"], description="Z axis"),
);

## 8. Cross-section: a slice through the canopy

From above, a canopy is a surface. A thin **slice** viewed side-on shows what the surface hides: crown shapes and depths, shrubs and grass **under** the trees, how far pulses got through the canopy to the ground, and gaps between crowns. This is **structural complexity**: the more layers and the more variation in height, the more complex the stand.

Move the slice with the sliders. The left panel shows where it sits on the canopy height model (CHM) raster from the catalogue.

In [ ]:
with rasterio.open(get_asset_href(asset_key="chm")) as src:
    window = from_bounds(LEFT, BOTTOM, RIGHT, TOP, src.transform)
    chm = src.read(1, window=window, boundless=True, fill_value=np.nan)
chm_extent = [LEFT, RIGHT, BOTTOM, TOP]

direction_w = widgets.ToggleButtons(options=["West-East", "South-North"], description="Direction")
position_w = widgets.FloatSlider(value=round((TOP - BOTTOM) / 2), min=0, max=round(TOP - BOTTOM),
                                 step=0.5, description="Position (m)", continuous_update=False)
width_w = widgets.FloatSlider(value=2.0, min=0.5, max=10, step=0.5, description="Width (m)",
                              continuous_update=False)


def update_position_range(change):
    position_w.max = round(TOP - BOTTOM) if change["new"] == "West-East" else round(RIGHT - LEFT)
direction_w.observe(update_position_range, names="value")


def slice_mask(direction, position, width):
    if direction == "West-East":      # a band of constant northing
        centre = BOTTOM + position
        return np.abs(pc["y"].values - centre) <= width / 2
    centre = LEFT + position          # a band of constant easting
    return np.abs(pc["x"].values - centre) <= width / 2


def show_slice(direction, position, width):
    sel = slice_mask(direction, position, width)
    along = (pc["x"].values[sel] - LEFT) if direction == "West-East" else (pc["y"].values[sel] - BOTTOM)
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(17, 5.5), width_ratios=[1, 2.4],
                                   constrained_layout=True)
    ax0.imshow(chm, cmap="viridis", extent=chm_extent, vmin=0, vmax=np.nanmax(chm))
    if direction == "West-East":
        band = Rectangle((LEFT, BOTTOM + position - width / 2), RIGHT - LEFT, width)
    else:
        band = Rectangle((LEFT + position - width / 2, BOTTOM), width, TOP - BOTTOM)
    band.set(fc="red", alpha=0.45, ec="red")
    ax0.add_patch(band)
    style_map(ax0, "Slice location on the CHM")
    cls = pc["classification"].values[sel]
    for code_ in (1, 2):
        k = cls == code_
        ax1.scatter(along[k], pc["z"].values[sel][k], s=0.3, c=CLASS_COLOURS[code_],
                    label=CLASS_NAMES[code_], rasterized=True)
    ax1.set_xlabel(f"Distance along slice, {direction.split('-')[0].lower()} to "
                   f"{direction.split('-')[1].lower()} (m)")
    ax1.set_ylabel("Elevation, Z (m)")
    ax1.set_title(f"Side view: {sel.sum():,} points in a {width:g} m slice",
                  fontsize=13, weight="bold")
    ax1.legend(markerscale=20, loc="upper right")
    plt.show()


out = widgets.interactive_output(show_slice, {"direction": direction_w, "position": position_w,
                                              "width": width_w})
display(widgets.VBox([widgets.HBox([direction_w, position_w, width_w]), out]))

**Look for:** crowns that look like one blob from above but are two trees side by side; shrubs beneath crowns; ground points under the canopy (pulses that got through); bare patches with nothing above the grass layer. Try a wide slice (10 m) and a narrow one (0.5 m): wider slices show more but overlap crowns.

## 9. Canopy height distribution and structural complexity

The **vertical profile** is the share of vegetation points in each height band: a summary of the side view for an area. One peak means a single layer; several peaks mean several strata. From the heights we compute a few common structural metrics:

| Metric | Meaning |
|---|---|
| **P95 height** | 95th percentile of vegetation heights: a robust "canopy top" that ignores stray high points |
| **Canopy cover** | % of first returns more than `COVER_HEIGHT` above the ground: how much of the sky the canopy blocks |
| **Height SD** | Standard deviation of vegetation heights: spread of the canopy vertically |
| **FHD** | Foliage height diversity: Shannon entropy of the vertical profile. Higher = points spread across more layers |

In [ ]:
COVER_HEIGHT = 2.0       # m
PROFILE_BIN = 0.5        # m
MIN_VEG_HEIGHT = 0.1     # m: points below this count as ground/surface, not vegetation
profile_edges = np.arange(0, np.ceil(pc["hag"].max()) + PROFILE_BIN, PROFILE_BIN)


def structure_metrics(df):
    veg = df["hag"].values[df["hag"].values >= MIN_VEG_HEIGHT]
    first = df["return_number"].values == 1
    profile, _ = np.histogram(veg, bins=profile_edges)
    p = profile[profile > 0] / profile.sum()
    return {
        "P95 height (m)": np.percentile(veg, 95),
        "Canopy cover (%)": 100 * np.mean(df["hag"].values[first] > COVER_HEIGHT),
        "Height SD (m)": veg.std(),
        "FHD": -(p * np.log(p)).sum(),
    }, profile / profile.sum()


slice_sel = slice_mask(direction_w.value, position_w.value, width_w.value)
metrics_aoi, profile_aoi = structure_metrics(pc)
metrics_slice, profile_slice = structure_metrics(pc[slice_sel])

centres = (profile_edges[:-1] + profile_edges[1:]) / 2
fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)
ax.plot(100 * profile_aoi, centres, "-o", color="#006d2c", label="Whole study area")
ax.plot(100 * profile_slice, centres, "-o", color="#d95f02", label="Current slice")
ax.set_xlabel("% of vegetation points")
ax.set_ylabel("Height above ground (m)")
ax.set_title("Vertical profile", fontsize=13, weight="bold")
ax.legend()
plt.show()

pd.DataFrame({"Whole study area": metrics_aoi, "Current slice": metrics_slice}).round(2)

### Structure as maps

The same metrics computed for every `GRID_RES` (1 m) cell turn the point cloud into rasters, ready to compare with other gridded data such as the vegetation indices from Chapter 2.

In [ ]:
GRID_RES = 1.0   # metres
NCOLS = int(np.ceil((RIGHT - LEFT) / GRID_RES))
NROWS = int(np.ceil((TOP - BOTTOM) / GRID_RES))
GRID_EXTENT = [LEFT, LEFT + NCOLS * GRID_RES, TOP - NROWS * GRID_RES, TOP]

# Index of the grid cell (row-major, north up) each point falls in
col = np.floor((pc["x"].values - LEFT) / GRID_RES).astype(int)
row = np.floor((TOP - pc["y"].values) / GRID_RES).astype(int)
inside = (col >= 0) & (col < NCOLS) & (row >= 0) & (row < NROWS)
pc["cell"] = np.where(inside, row * NCOLS + col, -1)

veg_pts = (pc["hag"].values >= MIN_VEG_HEIGHT) & inside
cells_veg = pc["cell"].values[veg_pts]
hag_veg = pc["hag"].values[veg_pts]
n_cells = NROWS * NCOLS

# P95 height per cell
p95 = (pd.Series(hag_veg).groupby(cells_veg).quantile(0.95)
       .reindex(range(n_cells)).values.reshape(NROWS, NCOLS))

# Canopy cover per cell: share of first returns above COVER_HEIGHT
first = pc["return_number"].values == 1
cells_first = pc["cell"].values[first]
above = pc["hag"].values[first] > COVER_HEIGHT
cover = 100 * (np.bincount(cells_first, weights=above, minlength=n_cells)
               / np.maximum(np.bincount(cells_first, minlength=n_cells), 1)).reshape(NROWS, NCOLS)

# FHD per cell: entropy of each cell's vertical profile
bins = np.clip(np.digitize(hag_veg, profile_edges) - 1, 0, len(profile_edges) - 2)
nb = len(profile_edges) - 1
hist = np.bincount(cells_veg * nb + bins, minlength=n_cells * nb).reshape(n_cells, nb).astype(float)
prop = hist / np.maximum(hist.sum(axis=1, keepdims=True), 1)
with np.errstate(divide="ignore", invalid="ignore"):
    fhd = -np.nansum(np.where(prop > 0, prop * np.log(prop), 0), axis=1).reshape(NROWS, NCOLS)
fhd[hist.sum(axis=1).reshape(NROWS, NCOLS) < 10] = np.nan   # too few vegetation points

fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
for ax, data, title, cmap, label in [
    (axes[0], p95, "P95 height", "viridis", "m"),
    (axes[1], cover, f"Canopy cover (> {COVER_HEIGHT:g} m)", "Greens", "%"),
    (axes[2], fhd, "Foliage height diversity", "magma", "FHD"),
]:
    im = ax.imshow(data, cmap=cmap, extent=GRID_EXTENT)
    fig.colorbar(im, ax=ax, label=label, shrink=0.85)
    style_map(ax, title)
plt.show()

In [ ]:
# Optional: save the 1 m structure grids as one GeoTIFF for GIS use.
EXPORT_GRIDS = True
GRID_PATH = OUTPUT_DIR / "lidar_structure_1m.tif"

if EXPORT_GRIDS:
    layers = {"P95_height_m": p95, "canopy_cover_pct": cover, "FHD": fhd}
    GRID_PATH.parent.mkdir(parents=True, exist_ok=True)
    profile = dict(driver="GTiff", width=NCOLS, height=NROWS, count=len(layers), dtype="float32",
                   crs="EPSG:7853", nodata=np.nan, compress="deflate",
                   transform=rasterio.transform.from_origin(LEFT, TOP, GRID_RES, GRID_RES))
    with rasterio.open(GRID_PATH, "w", **profile) as dst:
        for band, (name, data) in enumerate(layers.items(), start=1):
            dst.write(data.astype("float32"), band)
            dst.set_band_description(band, name)
    print(f"Wrote {GRID_PATH} with bands: {', '.join(layers)}")

## 10. Close and resources

**What we did**

1. Found the LiDAR in the STAC catalogue and told a COPC apart from a plain LAS by its header.
2. Read only the study area from a 339-million-point cloud, one octree level at a time.
3. Checked CRS, coverage and attributes.
4. Filtered by level, class, height and return.
5. Viewed the cloud in 3D, sliced it, and summarised structure as profiles and 1 m metric maps.

**Activities**

1. Set `Max level` to 3 in Section 6. Which features survive the thinning, and which disappear?
2. Find a slice position in Section 8 that crosses two strata (shrubs under trees). Rerun Section 9: how do the slice's FHD and profile differ from the whole area's?
3. Change `GRID_RES` to 2 or 5 m in Section 9 and rerun that cell. How do the P95, cover and FHD maps change, and what detail is lost?

**Resources**

- The survey's own LiDAR processing report: the `lidar_report` asset in the catalogue
- COPC specification: <https://copc.io>
- ASPRS LAS 1.4 specification: <https://www.asprs.org/divisions-committees/lidar-division/laser-las-file-format-exchange-activities>
- laspy documentation: <https://laspy.readthedocs.io>
- PDAL, the point data abstraction library: <https://pdal.io>
- *lidR*, an R book covering the same workflows in depth: <https://r-lidar.github.io/lidRbook/>
- TERN DroneScape data: <https://www.tern.org.au>